# ---------- Data loading notebook ----------

The goal of this notebook is to prepare London bike-sharing trips starting in the selected YEAR.

Author: Artur Werys
### Pamięć podczas przetwarzania

Uruchamiaj cały notebook po restarcie kernela. Po filtrowaniu usuwamy tabelę pośrednią. Nazwy stacji i modele rowerów przechowujemy jako kategorie; nie wykonujemy dodatkowych głębokich kopii po filtrowaniu i sortowaniu. Odległości i prędkości obliczamy partiami. Pełna tabela wynikowa nadal musi zmieścić się w RAM.


In [46]:
from pathlib import Path
import gc
import pandas as pd
import xml.etree.ElementTree as et
import numpy as np

In [ ]:
YEAR = 2025 


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for path in [current, *current.parents]:
        if (path / "raw_data").is_dir():
            return path
    raise FileNotFoundError("Could not find project root containing raw_data/")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "raw_data"
YEAR_DIR = DATA_DIR / str(YEAR)
STATIONS_FILE = DATA_DIR / "stations.xml"
PROCESSED_DATA_DIR = PROJECT_ROOT / "processed_data" / "ready_2022_2025"


## ---------- Loading CSV data function ----------


In [48]:
OLD_COLUMN_NAMES = {
    "Rental Id": "Number",
    "Start Date": "Start date",
    "StartStation Id": "Start station number",
    "StartStation Name": "Start station",
    "End Date": "End date",
    "EndStation Id": "End station number",
    "EndStation Name": "End station",
    "Bike Id": "Bike number",
}


def format_duration(seconds):
    """Use the day/hour/minute/second format of the modern CSV files."""
    days, remainder = divmod(int(seconds), 86400)
    hours, remainder = divmod(remainder, 3600)
    minutes, seconds = divmod(remainder, 60)
    if days:
        return f"{days}d {hours}h {minutes}m {seconds}s"
    if hours:
        return f"{hours}h {minutes}m {seconds}s"
    if minutes:
        return f"{minutes}m {seconds}s"
    return f"{seconds}s"


def load_data(folder_path):
    files = list(folder_path.glob("*.csv"))
    # A weekly extract can span New Year and live in the adjacent year's folder.
    for adjacent_year in (YEAR - 1, YEAR + 1):
        folder = folder_path.parent / str(adjacent_year)
        files.extend(folder.glob(f"*JourneyDataExtract*{YEAR}*.csv"))
    if not files:
        raise FileNotFoundError(f"No trip CSV files found for {YEAR} in {folder_path.parent}")

    dataframes = []
    for file in files:
        df = pd.read_csv(file, low_memory=False)
        if "Rental Id" in df.columns:
            df.rename(columns=OLD_COLUMN_NAMES, inplace=True)
            duration_seconds = pd.to_numeric(df.pop("Duration"), errors="coerce")
            df["Total duration (ms)"] = duration_seconds * 1000
            df["Total duration"] = duration_seconds.map(format_duration, na_action="ignore")
            df["Bike model"] = pd.NA
            # One July 2022 extract omits the end station ID.
            if "End station number" not in df.columns:
                df["End station number"] = np.nan
        # Select by start year in either source date format before counting or cleaning.
        # Leave missing or unrecognised dates to the original validation steps below.
        start_year = df["Start date"].astype("string").str.extract(
            r"^\s*(?:\d{1,2}/\d{1,2}/)?(\d{4})(?:[- T]|$)", expand=False
        )
        dataframes.append(df.loc[start_year.isna() | start_year.eq(str(YEAR))])

    combined_data = pd.concat(dataframes, ignore_index=True)

    return combined_data


## ---------- Loading data from the chosen year ----------


In [49]:
print(f"--- Loading data from {YEAR}... ---")

combined_trip_data = load_data(YEAR_DIR)

print(combined_trip_data.head())
print(f"Number of trips in {YEAR}:", len(combined_trip_data))


--- Loading data from 2025... ---
      Number        Start date  Start station number  \
0  145666816  2025-01-14 23:59                  1043   
1  145666817  2025-01-14 23:59                300015   
2  145666818  2025-01-14 23:59                  1068   
3  145666819  2025-01-14 23:59                  1159   
4  145666812  2025-01-14 23:58                200048   

                      Start station          End date  End station number  \
0        Museum of London, Barbican  2025-01-15 00:13            200149.0   
1          Binfield Road, Stockwell  2025-01-15 00:04            300229.0   
2  Norton Folgate, Liverpool Street  2025-01-15 00:10              1051.0   
3         Berry Street, Clerkenwell  2025-01-15 00:10              1007.0   
4          Page Street, Westminster  2025-01-15 00:14              1058.0   

                        End station  Bike number  Bike model Total duration  \
0           Watney Street, Shadwell        55223     CLASSIC        14m 30s   
1       

### ---------- Preparing the selected dataset ----------


In [50]:
# Repeated labels need only a category code per row.
for column in ["Start station", "End station", "Bike model"]:
    combined_trip_data[column] = combined_trip_data[column].astype("category")
gc.collect()


0

### ---------- Renaming columns ----------


In [51]:
combined_trip_data.rename(columns={
    "Number": "trip_id",
    "Start date": "start_date",
    "End date": "end_date",
    "Start station number": "start_station_id",
    "Start station": "start_station_name",
    "End station number": "end_station_id",
    "End station": "end_station_name",
    "Bike number": "bike_id",
    "Bike model": "bike_model",
    "Total duration": "total_duration",
    "Total duration (ms)": "total_duration_ms",
}, inplace=True)

combined_trip_data["total_duration_minutes"] = (
    pd.to_numeric(combined_trip_data["total_duration_ms"], errors="coerce") / 60000
)
combined_trip_data.drop(columns=["total_duration_ms"], inplace=True)

combined_trip_data.head()



,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_minutes
0,145666816,2025-01-14 23:59,1043,"Museum of London, Barbican",2025-01-15 00:13,200149.0,"Watney Street, Shadwell",55223,CLASSIC,14m 30s,14.507800
1,145666817,2025-01-14 23:59,300015,"Binfield Road, Stockwell",2025-01-15 00:04,300229.0,"Sidney Road, Stockwell",20325,CLASSIC,5m 7s,5.119683
2,145666818,2025-01-14 23:59,1068,"Norton Folgate, Liverpool Street",2025-01-15 00:10,1051.0,"Shoreditch Park, Hoxton",50432,CLASSIC,10m 40s,10.679250
3,145666819,2025-01-14 23:59,1159,"Berry Street, Clerkenwell",2025-01-15 00:10,1007.0,"Drury Lane, Covent Garden",56794,CLASSIC,10m 42s,10.709133
4,145666812,2025-01-14 23:58,200048,"Page Street, Westminster",2025-01-15 00:14,1058.0,"Edgware Road Station, Marylebone",63546,PBSC_EBIKE,15m 44s,15.736417


### ---------- Checking for missing values in combined data ----------


In [52]:
print("Missing values in each column:")
print(combined_trip_data.isna().sum())

len_before_dropping = len(combined_trip_data)

print("Dropping rows with missing core trip data...")

combined_trip_data = combined_trip_data.dropna(
    subset=["end_date", "end_station_id", "bike_id"]
)

print("Missing values in each column after dropping rows with missing core trip data:")
print(combined_trip_data.isna().sum())



Missing values in each column:
trip_id                     0
start_date                  0
start_station_id            0
start_station_name          0
end_date                  184
end_station_id            184
end_station_name          184
bike_id                     0
bike_model                  0
total_duration            184
total_duration_minutes    184
dtype: int64
Dropping rows with missing core trip data...
Missing values in each column after dropping rows with missing core trip data:
trip_id                   0
start_date                0
start_station_id          0
start_station_name        0
end_date                  0
end_station_id            0
end_station_name          0
bike_id                   0
bike_model                0
total_duration            0
total_duration_minutes    0
dtype: int64


## ---------- Loading station data from XML ----------


In [53]:
tree = et.parse(STATIONS_FILE)
root = tree.getroot()

stations_data = []

for station in root.findall("station"):

    station_data = [
        station.find("name").text.strip(),
        station.find("lat").text,
        station.find("long").text
    ]

    stations_data.append(station_data)

stations_df = pd.DataFrame(
    stations_data,
    columns=[
        "station_name",
        "latitude",
        "longitude"
    ]
)

print("--- Station data loaded from XML ---")
print(stations_df.head())
print("Number of stations:", len(stations_df))

--- Station data loaded from XML ---
                           station_name     latitude     longitude
0            River Street , Clerkenwell  51.52916347  -0.109970527
1        Phillimore Gardens, Kensington  51.49960695  -0.197574246
2  Christopher Street, Liverpool Street  51.52128377  -0.084605692
3       St. Chad's Street, King's Cross  51.53005939  -0.120973687
4         Sedding Street, Sloane Square     51.49313     -0.156876
Number of stations: 801


### ---------- Adding station coordinates ----------

Map coordinates by station name instead of merging the entire trip table twice.


In [54]:
station_coordinates = stations_df.set_index("station_name")[["latitude", "longitude"]].apply(
    pd.to_numeric, errors="coerce"
)
if not station_coordinates.index.is_unique:
    raise ValueError("Station names in stations.xml must be unique for coordinate mapping.")

for side in ["start", "end"]:
    station_names = combined_trip_data[f"{side}_station_name"]
    for coordinate, suffix in [("latitude", "lat"), ("longitude", "lon")]:
        combined_trip_data[f"{side}_{suffix}"] = station_names.map(
            station_coordinates[coordinate]
        ).astype("float64")
del station_names
combined_trip_data.head()


,trip_id,start_date,start_station_id,start_station_name,end_date,end_station_id,end_station_name,bike_id,bike_model,total_duration,total_duration_minutes,start_lat,start_lon,end_lat,end_lon
0,145666816,2025-01-14 23:59,1043,"Museum of London, Barbican",2025-01-15 00:13,200149.0,"Watney Street, Shadwell",55223,CLASSIC,14m 30s,14.507800,51.517821,-0.096497,51.511542,-0.056667
1,145666817,2025-01-14 23:59,300015,"Binfield Road, Stockwell",2025-01-15 00:04,300229.0,"Sidney Road, Stockwell",20325,CLASSIC,5m 7s,5.119683,51.472510,-0.122832,51.469202,-0.119023
2,145666818,2025-01-14 23:59,1068,"Norton Folgate, Liverpool Street",2025-01-15 00:10,1051.0,"Shoreditch Park, Hoxton",50432,CLASSIC,10m 40s,10.679250,51.521114,-0.078870,51.534043,-0.086380
3,145666819,2025-01-14 23:59,1159,"Berry Street, Clerkenwell",2025-01-15 00:10,1007.0,"Drury Lane, Covent Garden",56794,CLASSIC,10m 42s,10.709133,51.522853,-0.099994,51.514771,-0.122220
4,145666812,2025-01-14 23:58,200048,"Page Street, Westminster",2025-01-15 00:14,1058.0,"Edgware Road Station, Marylebone",63546,PBSC_EBIKE,15m 44s,15.736417,51.493978,-0.127554,NaN,NaN


## ---------- Sorting rides in time for each bike ----------


In [55]:
def parse_trip_dates(values):
    """Parse ISO and British CSV timestamps without swapping month and day."""
    if pd.api.types.is_datetime64_any_dtype(values):
        return values
    text = values.astype("string").str.strip()
    iso = text.str.match(r"^\d{4}-\d{2}-\d{2}(?:[ T]|$)", na=False)
    british = text.str.match(r"^\d{1,2}/\d{1,2}/\d{4}(?:[ T]|$)", na=False)
    unknown = text.notna() & text.ne("") & ~(iso | british)
    if unknown.any():
        raise ValueError(f"Unsupported trip date format: {text[unknown].head().tolist()}")
    result = pd.Series(pd.NaT, index=values.index, dtype="datetime64[ns]")
    result.loc[iso] = pd.to_datetime(text[iso], format="ISO8601", errors="raise")
    result.loc[british] = pd.to_datetime(
        text[british], format="mixed", dayfirst=True, errors="raise"
    )
    return result


# Parse dates and coordinates before filtering and chronological sorting.
for date_column in ["start_date", "end_date"]:
    combined_trip_data[date_column] = parse_trip_dates(combined_trip_data[date_column])

coordinate_columns = ["start_lat", "start_lon", "end_lat", "end_lon"]
for coordinate_column in coordinate_columns:
    combined_trip_data[coordinate_column] = pd.to_numeric(
        combined_trip_data[coordinate_column], errors="coerce"
    )

final_trip_data = combined_trip_data.dropna(
    subset=["start_date", "end_date", *coordinate_columns]
)
# The filtered table owns the remaining processing pipeline.
del combined_trip_data
gc.collect()


16

In [56]:
# Some retired stations have alphanumeric IDs. After the original coordinate
# filter, the remaining IDs fit the numeric types of the 2024–2025 result.
final_trip_data = final_trip_data.astype({
    "bike_id": "float64",
    "start_station_id": "int64",
    "end_station_id": "float64",
})

# sort_values already returns a table; a deep copy would duplicate its blocks.
final_trip_data = final_trip_data.sort_values(
    ["bike_id", "start_date", "end_date"], kind="stable", ignore_index=True
)
gc.collect()


0

## ---------- How much data was removed? ----------

In [57]:
len_after_dropping = len(final_trip_data)
dropped_rows_count = len_before_dropping - len_after_dropping

dropped_percent = dropped_rows_count / len_before_dropping * 100
remaining_percent = len_after_dropping / len_before_dropping * 100

print(f"Number of rows before dropping: {len_before_dropping:.2e}")
print(f"Number of rows after dropping: {len_after_dropping:.2e}")
print(f"Number of dropped rows: {dropped_rows_count:.2e}")

print(f"Percentage of dropped rows: {dropped_percent:.3f}%")
print(f"Percentage of remaining rows: {remaining_percent:.3f}%")



Number of rows before dropping: 9.07e+06
Number of rows after dropping: 8.91e+06
Number of dropped rows: 1.62e+05
Percentage of dropped rows: 1.784%
Percentage of remaining rows: 98.216%


## ---------- Saving final trip data as parquet ----------


In [58]:
# Encode the seven weekday names without creating millions of Python strings.
final_trip_data["start_day_of_week"] = pd.Categorical.from_codes(
    final_trip_data["start_date"].dt.dayofweek.to_numpy(dtype=np.int8),
    categories=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
)


def add_distance_and_speed(trips, batch_size=250_000):
    """Bound temporary trigonometric arrays to one batch of trips."""
    distances = np.empty(len(trips), dtype=np.float64)
    speeds = np.full(len(trips), np.nan, dtype=np.float64)
    for start in range(0, len(trips), batch_size):
        stop = min(start + batch_size, len(trips))
        block = trips.iloc[start:stop]
        start_lat = np.radians(block["start_lat"].to_numpy())
        start_lon = np.radians(block["start_lon"].to_numpy())
        end_lat = np.radians(block["end_lat"].to_numpy())
        end_lon = np.radians(block["end_lon"].to_numpy())
        haversine_a = (
            np.sin((end_lat - start_lat) / 2) ** 2
            + np.cos(start_lat) * np.cos(end_lat) * np.sin((end_lon - start_lon) / 2) ** 2
        )
        distances[start:stop] = 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(haversine_a, 0, 1)))
        hours = block["total_duration_minutes"].to_numpy() / 60
        np.divide(distances[start:stop], hours, out=speeds[start:stop], where=hours > 0)
    trips["straight_line_distance_km"] = distances
    trips["speed_km_per_h"] = speeds


add_distance_and_speed(final_trip_data)
final_trip_data = final_trip_data.reindex(sorted(final_trip_data.columns), axis=1)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
final_trip_data.to_parquet(PROCESSED_DATA_DIR / f"final_trip_data_{YEAR}.parquet")


In [59]:
final_trip_data.head()

,bike_id,bike_model,end_date,end_lat,end_lon,end_station_id,end_station_name,speed_km_per_h,start_date,start_day_of_week,start_lat,start_lon,start_station_id,start_station_name,straight_line_distance_km,total_duration,total_duration_minutes,trip_id
0,2.0,CLASSIC,2025-01-14 18:27:00,51.497304,-0.022793,200109.0,"Alpha Grove, Millwall",8.042234,2025-01-14 18:03:00,Tuesday,51.508622,-0.065006,200090,"Pennington Street, Wapping",3.181329,23m 44s,23.734667,145660049
1,2.0,CLASSIC,2025-01-14 21:07:00,51.505703,-0.027773,300228.0,"Westferry Circus, Canary Wharf",9.712075,2025-01-14 21:00:00,Tuesday,51.497304,-0.022793,200109,"Alpha Grove, Millwall",0.995509,6m 9s,6.150133,145665363
2,2.0,CLASSIC,2025-01-14 21:52:00,51.501513,-0.123823,300249.0,"Westminster Pier, Westminster",10.431536,2025-01-14 21:14:00,Tuesday,51.505703,-0.027773,300228,"Westferry Circus, Canary Wharf",6.664447,38m 19s,38.332500,145665536
3,2.0,CLASSIC,2025-01-14 22:36:00,51.497304,-0.022793,200109.0,"Alpha Grove, Millwall",9.840540,2025-01-14 21:54:00,Tuesday,51.501513,-0.123823,300249,"Westminster Pier, Westminster",7.009102,42m 44s,42.736083,145666019
4,2.0,CLASSIC,2025-01-17 14:40:00,51.503400,-0.020242,200163.0,"Jubilee Plaza, Canary Wharf",8.874631,2025-01-17 14:35:00,Friday,51.497304,-0.022793,200109,"Alpha Grove, Millwall",0.700462,4m 44s,4.735717,145727708


In [60]:
final_trip_data.tail(5)


,bike_id,bike_model,end_date,end_lat,end_lon,end_station_id,end_station_name,speed_km_per_h,start_date,start_day_of_week,start_lat,start_lon,start_station_id,start_station_name,straight_line_distance_km,total_duration,total_duration_minutes,trip_id
8906439,99980.0,CLASSIC,2025-04-20 10:55:00,51.532200,-0.105481,1060.0,"Torrens Street, Angel",0.000000,2025-04-20 10:55:00,Sunday,51.532200,-0.105481,1060,"Torrens Street, Angel",0.000000,9s,0.162567,147901262
8906440,99981.0,CLASSIC,2025-09-21 15:35:00,51.527152,-0.058005,200156.0,"Pott Street, Bethnal Green",0.000000,2025-09-21 15:35:00,Sunday,51.527152,-0.058005,200156,"Pott Street, Bethnal Green",0.000000,18s,0.307950,152337403
8906441,99984.0,CLASSIC,2025-01-19 15:29:00,51.452997,-0.148652,300006.0,"Clapham South, Clapham South",2.016572,2025-01-19 12:04:00,Sunday,51.506585,-0.199004,3487,"Campden Hill Road, Notting Hill",6.903956,3h 25m 24s,205.416633,145759306
8906442,99987.0,CLASSIC,2025-02-16 09:13:00,51.496791,-0.093913,1208.0,"Dickens Square, Borough",3.352350,2025-02-16 09:04:00,Sunday,51.496520,-0.101384,1200,"Ontario Street, Elephant & Castle",0.518035,9m 16s,9.271733,146335518
8906443,99992.0,CLASSIC,2025-01-18 14:07:00,51.536384,-0.102758,300031.0,"Islington Green, Angel",0.000000,2025-01-18 14:04:00,Saturday,51.536384,-0.102758,300031,"Islington Green, Angel",0.000000,2m 53s,2.898483,145746570
